In [105]:
import os, sys

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
project_root
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root already in sys.path


In [106]:
# imports
import numpy as np
import pandas as pd
from utils import gro_processing

# Get data
df = gro_processing.read_gro("../../data/npt-HK4.gro")[0]
df

,res_id,res_name,atom_name,atom_id,x,y,z,Vx,Vy,Vz
0,1,HK4,H28,1,1.602,0.962,0.912,-0.0935,0.8795,-2.1573
1,1,HK4,C48,2,1.565,0.879,0.852,-0.2814,-0.6983,0.0954
2,1,HK4,C47,3,1.655,0.809,0.773,0.4154,0.1398,0.1445
3,1,HK4,H27,4,1.754,0.855,0.756,0.1277,0.7999,0.2091
4,1,HK4,C46,5,1.603,0.717,0.683,-0.0655,0.7587,-0.2231
...,...,...,...,...,...,...,...,...,...,...
126079,1501,HK4,H23,126080,0.953,5.984,1.494,-1.2440,0.1068,-0.7226
126080,1501,HK4,C44,126081,0.736,6.009,1.525,0.1513,0.5200,0.5632
126081,1501,HK4,H24,126082,0.707,6.041,1.425,0.3100,2.4172,1.1123
126082,1501,HK4,C45,126083,0.632,5.988,1.615,-0.0110,-0.1637,0.2235


In [107]:
select_range = list(range(1,3 + 1))

df = df[df["res_id"].isin(select_range)]

In [114]:
"""
If say I have 84 atoms,
and I want to select some to be the "electron clump"
then I would want to input a range or specific id of the atom,
so say like I want atom 24:56 and 80 and 84 to be my clump

Then from there I would want to parse the entire df, to only have data of the atoms of the respective id
and for every molecule as well
"""

num_atoms_one_mol = df.loc[df["res_id"] == 1].shape[0]
num_res = int(df.iloc[-1]["res_id"])

# print("Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' ")
# user_input = input("Enter atom id:")
user_input = "15-20; 30; 31-33"

#formatting user_input:
split_user_input = user_input.split("; ")
try:
    indices = [
        parts if len(parts := tuple(map(int, i.split("-")))) > 1 else int(i)
        for i in split_user_input
    ]  # not sure how to make this more readable but i like it in one line
except: print("Incorrect formatting, please follow the instructions above.")


new_df = pd.DataFrame()
for res in range(0,num_res):
    for i in indices:
        # in case range:
        if type(i) == tuple: 
            for id in range(i[0], i[1]+1):
                temp = df.loc[df["atom_id"] == (id + (num_atoms_one_mol * res))]
                new_df = pd.concat([new_df,temp])
        else: 
            temp = df.loc[df["atom_id"] == (i + (num_atoms_one_mol * res))]
            new_df = pd.concat([new_df,temp])        

In [115]:
new_df.shape

(30, 10)